# Finalizing metafile for ADDL pipeline
### - Attaching links of according brain scans to metafile
### - Selecting MRI and amyloid scans with needed proccessing

In [2]:
# import libraries
import numpy as np 
import pandas as pd 
import os, fnmatch
import sys

In [3]:
#pd.set_option('display.max_rows', 20)
#pd.set_option('display.max_columns', None)

In [4]:
# import data
#meta = pd.read_csv('ADNI_A4_processed_9_27_2023.csv')
meta = pd.read_csv('ADNI_A4_processed_02_06_2024.csv', index_col=[0],low_memory=False)

In [5]:
meta.shape

(230151, 18)

In [6]:
meta

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,Visit,Study.Date,Archive.Date,Age,Modality,Description,Type,Imaging.Protocol,Image.ID,VISCODE,Image.Data.ID,modality_subtype
1,002_S_0295,ADNI,ADNI 1,M,74.5,CN,ADNI Screening,4/18/2006,4/18/2006,84.9,MRI,B1-Calibration Body,Original,Acquisition Plane=SAGITTAL;Slice Thickness=2.5...,13710,sc,I13710,B1-Calibration Body
2,002_S_0295,ADNI,ADNI 1,M,74.5,CN,ADNI Screening,4/18/2006,4/18/2006,84.9,MRI,B1-Calibration PA,Original,Acquisition Plane=SAGITTAL;Slice Thickness=2.5...,13711,sc,I13711,B1-Calibration PA
3,002_S_0295,ADNI,ADNI 1,M,74.5,CN,ADNI Screening,4/18/2006,4/18/2006,84.9,MRI,3-plane localizer,Original,Acquisition Plane=AXIAL;Slice Thickness=5.0;Ma...,13712,sc,I13712,3-plane localizer
4,002_S_0295,ADNI,ADNI 1,M,74.5,CN,ADNI Screening,4/18/2006,4/18/2006,84.9,MRI,3-plane localizer,Original,Acquisition Plane=CORONAL;Slice Thickness=5.0;...,13713,sc,I13713,3-plane localizer
5,002_S_0295,ADNI,ADNI 1,M,74.5,CN,ADNI Screening,4/18/2006,4/18/2006,84.9,MRI,3-plane localizer,Original,Acquisition Plane=SAGITTAL;Slice Thickness=5.0...,13714,sc,I13714,3-plane localizer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230147,B99968206,A4,NaN,M,100.0,LEARN amyloidNE,SCV4,1/01/2016,11/18/2020,0.0,MRI,T2_star <- AXIAL T2-STAR,Processed,NaN,1372799,SCV4,I1372799,T2
230148,B99968206,A4,NaN,M,100.0,LEARN amyloidNE,SCV4,1/01/2016,12/08/2020,0.0,MRI,FLAIR; DeFaced <- AXIAL T2W-FLAIR,Processed,NaN,1383111,SCV4,I1383111,FLAIR
230149,B99968206,A4,NaN,M,100.0,LEARN amyloidNE,SCV4,1/01/2016,12/09/2020,0.0,MRI,T1; DeFaced <- SAG ACCELERATED MPRAGE,Processed,NaN,1384729,SCV4,I1384729,T1
230150,B99968206,A4,NaN,M,100.0,LEARN amyloidNE,SCV4,1/01/2016,12/08/2020,0.0,MRI,T2_SE; DeFaced <- AXIAL T2-TSE WITH FAT SAT,Processed,NaN,1383968,SCV4,I1383968,T2


In [4]:
#meta[meta.Project == 'ADNI']

In [5]:
# don't forget to check the path everywhere, it should lead to the right folder!!!
#os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/')

In [6]:
fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/'), '*I1321166.nii')

['A4_B10018169_MR_Florbetapir_Br_20200720182306561_S893044_I1321166.nii']

In [7]:
fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/'), '*I13724.nii')

[]

In [8]:
fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/'), '*I1619403.nii')

['I1619403.nii']

### run on cluster if the file is big 

In [67]:
for i in range(1,np.shape(meta)[0]+1):
    name = '*' + meta['Image.Data.ID'][i]+'.nii'
    filename = fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/'), name)
    if len(filename)> 0:
        meta.loc[i,'PATH'] = '/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/' + filename[0]
        
    print(i)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


KeyboardInterrupt: 

In [ ]:
# removing empty lines, scans that were not downloaded

In [15]:
meta['PATH'].isnull().values.any()

True

In [16]:
meta['PATH'].isnull().sum()

37411

In [17]:
meta2 = meta[meta['PATH'].notna()]

In [18]:
meta2['PATH'].isnull().values.any()

False

In [19]:
np.shape(meta2)

(31431, 26)

In [20]:
# save metafile
meta2.to_csv('metafile_completed_ADNI_A4_processed_02_06_2024.csv')

In [7]:
sys.exit("I am done")

SystemExit: 

/csc/epitkane/home/atagmazi/.conda/envs/atagmazi_gpu5/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [18]:
# upload metafile, if part of the script was run on the cluster
meta2 = pd.read_csv('metafile_completed_ADNI_A4_processed_02_06_2024.csv')

In [19]:
meta2.shape

(65280, 19)

In [20]:
meta2.PATH[111]

'/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23/ADNI_002_S_0559_MR_MPR__GradWarp__B1_Correction__N3_Br_20070217024753319_S23905_I40807.nii'

In [21]:
#meta2

In [22]:
meta2.Phase.value_counts()

ADNI 2     23687
ADNI 1     17040
ADNI 3      8741
ADNI GO     3195
Name: Phase, dtype: int64

In [23]:
#pd.set_option('display.max_rows', None)
#meta2.modality_subtype.value_counts()

In [24]:
# selecting all MRI and only amyloid PET scans
pd.set_option('display.max_rows', 20)
meta3 = meta2[(meta2.Modality == 'MRI') | 
      (meta2.modality_subtype == 'AV45')|
     (meta2.modality_subtype == 'FBB')|
     (meta2.modality_subtype == 'PIB')]

In [25]:
meta.Project.value_counts()

ADNI    217542
A4       12609
Name: Project, dtype: int64

In [26]:
meta2.Type.value_counts()

Processed    44799
Original     20481
Name: Type, dtype: int64

In [27]:
meta2.Project.value_counts()

ADNI    52671
A4      12609
Name: Project, dtype: int64

In [28]:
meta3.Project.value_counts()

ADNI    42798
A4      12162
Name: Project, dtype: int64

In [29]:
#pd.set_option('display.max_rows', None)
#meta3[meta3.Project == 'A4'].Description.value_counts()

In [30]:
# selecting only T1 and amyloid PET scans with needed pre-proccesing 
meta4 = meta3[((meta3.Project == 'ADNI')&((meta3.Description.str.contains('MP*RAGE|T1|IR*SPGR')) | 
     (meta3.Description.str.contains('Co-registered, Averaged'))))|
           (( meta3.Project == 'A4')&((meta3.Description.str.contains('T1*')) | 
     (meta3.Description.str.contains('Flor*'))))]

In [31]:
meta4.Project.value_counts()

ADNI    31260
A4      11655
Name: Project, dtype: int64

In [14]:
#meta4[meta4.Subject.ID.str.contains('941_S_6080')]

In [62]:
#meta4[meta4['Subject.ID'] == '003_S_6158']

,Unnamed: 0.1,Unnamed: 0,Subject.ID,Project,Sex,Weight,Research.Group,APOE.A1,APOE.A2,Visit,...,Structure,Laterality,Image.Type,Registration,Tissue,VISCODE,Image.Data.ID,modality_subtype,PATH,year
2090,8266,8267,003_S_6158,ADNI,M,63.5,CN,3.0,3.0,ADNI Baseline,...,Brain,Both,image volume,native,All,bl,I974648,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2011


In [13]:
#pd.set_option('display.max_rows', None)
#meta4[meta4.Description.str.contains('T1')].Description.value_counts()

In [12]:
#pd.set_option('display.max_rows', None)
#meta4.Description.value_counts()

In [11]:
#meta4[meta4.Modality == 'PET'].Description.value_counts()

In [10]:
#meta4[meta4.Modality == 'MRI'].Description.value_counts()

In [32]:
meta4.Modality.value_counts()

MRI    34700
PET     8215
Name: Modality, dtype: int64

In [33]:
#save finale metafile for ADDL pipeline 
meta4.to_csv('metafile_ADDLpipeline_abeta_mri_02_06_2024.csv') # from metafile_completed_ADNI_A4_processed_9_27_2023.csv